In [34]:
import os
import random
import sys
from io import BytesIO

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn import metrics
from PIL import Image, ImageChops
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

In [35]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv

    CACHE_DIR = None

    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

In [36]:
seed = 100
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [37]:
DEFAULT_MEAN = (0.5, 0.5, 0.5)
DEFAULT_STD = (0.5, 0.5, 0.5)
IMAGE_SIZE = (224, 224)

# 1. Load the dataset from Hugging Face
if IN_COLAB:
    print(f'Loading dataset. Colab detected: using Drive cache at {CACHE_DIR}')
else:
    print('Loading dataset. Local environment detected: using default HF cache.')

test_data = load_dataset(
    'TheKernel01/140k-Real-and-Fake-Faces',
    split='test',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)

generator_mapping = {
    1: 'StyleGAN',
}

Loading dataset. Local environment detected: using default HF cache.


In [38]:
def sim_auc(scores, datasets):
    """
    Calculate AUC and FPR95 for multiple OOD datasets against ID dataset.
    """
    if len(scores) != len(datasets):
        raise ValueError('Number of scores arrays must match number of dataset names')

    if len(scores) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    scores = np.array(scores, dtype=object)
    id_confi = scores[0]

    auc_scores = []
    fpr_scores = []

    for ood_confi, dataset in zip(scores[1:], datasets[1:]):
        auroc, fpr_95 = calculate_auc_metrics(id_confi, ood_confi)
        auc_scores.append(auroc)
        fpr_scores.append(fpr_95)
        print(f'Dataset: {dataset:<25} | AUC: {auroc:.4f} | FPR95: {fpr_95:.4f}')

    avg_auc = np.mean(auc_scores)
    avg_fpr = np.mean(fpr_scores)

    print('-' * 60)
    print(f'Average AUC: {avg_auc:.4f} | Average FPR95: {avg_fpr:.4f}')

    return avg_auc, avg_fpr


def sim_ap(scores, datasets):
    """
    Calculate Average Precision for multiple OOD datasets against ID dataset.
    """
    if len(scores) != len(datasets):
        raise ValueError('Number of scores arrays must match number of dataset names')

    if len(scores) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    scores = np.array(scores, dtype=object)
    id_confi = scores[0]

    ap_scores = []

    for ood_confi, dataset in zip(scores[1:], datasets[1:]):
        aver_p = calculate_average_precision(id_confi, ood_confi)
        ap_scores.append(aver_p)
        print(f'Dataset: {dataset:<25} | AP: {aver_p:.4f}')

    avg_ap = np.mean(ap_scores)
    print('-' * 40)
    print(f'Average AP: {avg_ap:.4f}')

    return avg_ap


def calculate_auc_metrics(id_conf, ood_conf):
    # Combine predictions and create labels
    all_conf = np.concatenate([id_conf, ood_conf])
    # ID (Real) samples are positive (1), OOD (Fake) samples are negative (0)
    labels = np.concatenate([np.ones(len(id_conf)), np.zeros(len(ood_conf))])

    fpr, tpr, _ = metrics.roc_curve(labels, all_conf)
    auroc = metrics.auc(fpr, tpr)

    tpr_threshold = 0.95
    valid_indices = tpr >= tpr_threshold
    if np.any(valid_indices):
        fpr_at_95 = fpr[np.argmax(valid_indices)]
    else:
        fpr_at_95 = fpr[-1]
        print(f'Warning: 95% TPR not achievable. Max TPR: {tpr[-1]:.3f}')

    return auroc, fpr_at_95


def calculate_average_precision(id_predictions, ood_predictions):
    all_predictions = np.concatenate([id_predictions, ood_predictions])
    labels = np.concatenate(
        [np.ones(len(id_predictions)), np.zeros(len(ood_predictions))]
    )
    return metrics.average_precision_score(labels, all_predictions)

In [39]:
# 2. ELA Preprocessing and Custom PyTorch Dataset
def get_ela_image(image, quality=90):
    original = image.convert('RGB').resize(IMAGE_SIZE)
    buffer = BytesIO()
    original.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    compressed = Image.open(buffer)

    ela_img = ImageChops.difference(original, compressed)
    extrema = ela_img.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff

    return Image.fromarray(np.uint8(np.array(ela_img) * scale))

In [40]:
class HFELADataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        item = self.hf_data[idx]

        # Apply ELA processing
        ela_image = get_ela_image(item['image'])
        label = item['label']

        if self.transform:
            ela_image = self.transform(ela_image)

        return ela_image, label

In [41]:
class ELA_Detector:
    def __init__(self, checkpoint_path='./checkpoints/ela_detector_epoch5.pth'):
        self.model = models.resnet18(weights=None)
        self.model.fc = torch.nn.Linear(self.model.fc.in_features, 2)

        if os.path.exists(checkpoint_path):
            print(f'Loading model weights from {checkpoint_path}')
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
            self.model.load_state_dict(checkpoint['model_state_dict'])
        else:
            print(
                f'WARNING: Checkpoint {checkpoint_path} not found. Using randomly initialized weights.'
            )

        self.model = self.model.cuda()
        self.model.eval()

    @torch.no_grad()
    def detect(self, data):
        """
        Returns the probability of the image being REAL (Class 0).
        This matches the ID/OOD setup where ID (Real) needs higher scores than OOD (Fake).
        """
        logits = self.model(data)
        probs = F.softmax(logits, dim=1)
        real_probs = probs[:, 0]
        return real_probs

In [42]:
transform_ELA = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=DEFAULT_MEAN, std=DEFAULT_STD),
    ]
)

In [43]:
# 3. Filter the dataset into Real (ID) and specific AI-generated (OOD) subsets
all_generators = np.array(test_data['generator'])

# 3a. Extract Real (ID) dataset
real_indices = np.nonzero(all_generators == 0)[0]
real_hf = test_data.select(real_indices)
real_dataset = HFELADataset(real_hf, transform=transform_ELA)

evaluation_datasets = [('Real (ID)', real_dataset)]

# 3b. Extract Fake (OOD) datasets
for gen_id, gen_name in generator_mapping.items():
    fake_indices = np.nonzero(all_generators == gen_id)[0]
    fake_hf = test_data.select(fake_indices)
    fake_dataset = HFELADataset(fake_hf, transform=transform_ELA)
    evaluation_datasets.append((f'{gen_name} (OOD)', fake_dataset))

test_datasets = [name for name, _ in evaluation_datasets]

batch_size = 128 if IN_COLAB else 32

ela_detector = ELA_Detector(checkpoint_path='./checkpoints/ela_detector_epoch_5.pth')

In [44]:
with torch.no_grad():
    model_scores = []

    for dataset_name, dataset_obj in evaluation_datasets:
        data_loader = DataLoader(
            dataset_obj,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
        )

        dataset_scores = []
        total_num = 0

        for i, (samples, _) in enumerate(data_loader):
            samples = samples.cuda()
            samples_num = len(samples)
            total_num += samples_num

            scores = ela_detector.detect(samples)
            dataset_scores.append(scores)

            if total_num >= 1000:
                break

        if len(dataset_scores) > 0:
            dataset_scores = torch.cat(dataset_scores, dim=0)
            print(
                f'{dataset_name:<25}, Image number: {dataset_scores.shape[0]}, mean realness score: {dataset_scores.mean().item():.4f}'
            )
            model_scores.append(dataset_scores.cpu().numpy())
        else:
            print(f'Warning: {dataset_name} is empty. Check your label filtering!')

    print('\n' + '=' * 60)
    print('Detection Results AUC (Per Generator):')
    print('=' * 60)
    sim_auc(model_scores, test_datasets)

    print('\n' + '=' * 60)
    print('Detection Results AP (Per Generator):')
    print('=' * 60)
    sim_ap(model_scores, test_datasets)

Real (ID)                , Image number: 1024, mean realness score: 0.7183
StyleGAN (OOD)           , Image number: 1024, mean realness score: 0.7174

Detection Results AUC (Per Generator):
Dataset: StyleGAN (OOD)            | AUC: 0.5744 | FPR95: 0.9248
------------------------------------------------------------
Average AUC: 0.5744 | Average FPR95: 0.9248

Detection Results AP (Per Generator):
Dataset: StyleGAN (OOD)            | AP: 0.5666
----------------------------------------
Average AP: 0.5666
